# Fold-In — Personal User Vector
Hold the SVD item vectors fixed and solve for a personal user vector via least-squares using the 35 MovieLens-matched ratings.

SVD predicts a rating as: `r = global_mean + b_u + b_i + u · q_i`  
Known: `global_mean`, `b_i` (item biases), `q_i` (item vectors), `r` (personal ratings)  
Unknown: `u` (user vector, length k=20), `b_u` (user bias, scalar)

Rearranging: `r - global_mean - b_i ≈ b_u + u · q_i`  
This is a linear system — stack all 35 equations and solve with least-squares.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

data_path = Path('../data')

## Load item vectors and personal ratings

In [ ]:
d = np.load(data_path / 'svd_item_vectors.npz')
item_vectors = d['item_vectors']          # (n_items, k)
item_biases  = d['item_biases']           # (n_items,)
movieids     = d['movieids']              # (n_items,)
global_mean  = d['global_mean'][0]        # scalar

movieid_to_idx = {mid: i for i, mid in enumerate(movieids)}

print(f"Item matrix: {item_vectors.shape}")
print(f"Global mean: {global_mean:.4f}")

matched = pd.read_csv(data_path / 'movielens_matched.csv')
print(f"Personal matched ratings: {len(matched)}")
matched[['letterboxd_title', 'movieId', 'rating']].head()

## Build the least-squares system

In [ ]:
rows = []
for _, row in matched.iterrows():
    idx = movieid_to_idx.get(int(row['movieId']))
    if idx is None:
        continue
    qi    = item_vectors[idx]
    bi    = item_biases[idx]
    r     = row['rating']
    title = row['letterboxd_title']
    rows.append((qi, bi, r, title))

print(f"Ratings usable for fold-in: {len(rows)}")

# Target: r - global_mean - b_i
# Features: [1, q_i]  (1 for the user bias term)
A = np.column_stack([np.ones(len(rows)), np.array([row[0] for row in rows])])  # (n, k+1)
b = np.array([row[2] - global_mean - row[1] for row in rows])                  # (n,)

print(f"A shape: {A.shape}  (equations × unknowns)")

## Solve for user vector

In [ ]:
solution, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)

user_bias   = solution[0]
user_vector = solution[1:]

print(f"User bias (b_u): {user_bias:.4f}")
print(f"User vector shape: {user_vector.shape}")
print(f"User vector norm: {np.linalg.norm(user_vector):.4f}")

## Sanity check — predict back the training ratings

In [ ]:
preds, actuals, titles = [], [], []
for (qi, bi, r, title) in rows:
    pred = global_mean + user_bias + bi + user_vector @ qi
    preds.append(pred)
    actuals.append(r)
    titles.append(title)

results = pd.DataFrame({'title': titles, 'actual': actuals, 'predicted': preds})
results['error'] = results['predicted'] - results['actual']
rmse = np.sqrt((results['error'] ** 2).mean())
print(f"Train RMSE (fold-in): {rmse:.4f}")
results.sort_values('error', key=abs, ascending=False)

## Alternative: Ridge regression

Plain least-squares ignores the regularization used during SVD training (`reg=0.1`).
Ridge adds an L2 penalty `λ||u||²` that keeps the user vector in the same scale as the item vectors.

Solves: `(AᵀA + λI)x = Aᵀb`  via [`sklearn.linear_model.Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).
`λ = 0.1` matches the training regularization strength.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=0.1, fit_intercept=True)
ridge.fit(np.array([row[0] for row in rows]), b)

user_bias_ridge   = ridge.intercept_
user_vector_ridge = ridge.coef_

print(f"Ridge  — user bias: {user_bias_ridge:.4f}  |  vector norm: {np.linalg.norm(user_vector_ridge):.4f}")
print(f"lstsq  — user bias: {user_bias:.4f}  |  vector norm: {np.linalg.norm(user_vector):.4f}")

# Fold-in RMSE for Ridge
preds_ridge = [
    global_mean + user_bias_ridge + row[1] + user_vector_ridge @ row[0]
    for row in rows
]
rmse_ridge = np.sqrt(np.mean((np.array(preds_ridge) - np.array(actuals)) ** 2))
print(f"\nFold-in RMSE — lstsq: {rmse:.4f}  |  Ridge: {rmse_ridge:.4f}")